# Rotax 914 ULF - Step 4: Is it the engine, or a sensor?

**In one sentence:** Checks each of the 12 sensors separately and says whether it is working or is biased, drifting, stuck, spiking, noisy or dropping out - so a broken instrument is not mistaken for a failing engine.

| Goes in | Comes out |
|---|---|
| The same 128-second window | For each of 12 sensors, one of 7 conditions |

Starts from what step 2 (*Which part is failing?*) already learned.

Run the cells top to bottom with **Shift+Enter**. If this step was already trained with `run.py`, training just loads the saved model in a second; otherwise it trains here.

### 0. Setup

In [ ]:
# Finds the project folder wherever Jupyter was started, then loads everything.
import os, sys
d = os.path.abspath('.')
for _ in range(6):
    if os.path.exists(os.path.join(d, 'nb_helpers.py')): break
    if os.path.exists(os.path.join(d, 'validation_v4', 'nb_helpers.py')):
        d = os.path.join(d, 'validation_v4'); break
    d = os.path.dirname(d)
sys.path.insert(0, d)
from nb_helpers import *

ENGINE, PHASE = '914', 'sensor_fault'
setup(ENGINE, PHASE)

### 1. How often sensors fail in the data
Sensors are fine almost all the time. The rare failures come in six kinds,
each with its own signature - a steady offset, a slow drift, a frozen value,
sudden spikes, extra noise, or readings dropping to zero.

In [ ]:
flags = load_rows(ENGINE, P.SF_FLAG_COLS, split="train").to_numpy().ravel()
share = pd.Series(np.bincount(flags, minlength=len(P.SENSOR_FAULT_TYPES)),
                  index=P.SENSOR_FAULT_TYPES) / len(flags)
share.drop("none").mul(100).plot.barh(color=FAULTY, figsize=(7, 3))
plt.xlabel("% of sensor readings with this fault")
plt.title(f"Sensors are fine {share['none']:.1%} of the time")
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

### 2. Train the model (or load it)

In [ ]:
payload = train_or_load(ENGINE, PHASE)
print(f"Done. Trained for {payload.get('epochs_run', '?')} passes over the data.")

### 3. Scorecard
Measured on the test flights. The quality bar catches a model that did not learn - it is a safety check, not a target.

In [ ]:
scorecard(ENGINE, PHASE)

### 4. See it working on flights it never saw
The share of sensor channels identified correctly, split by what was really
wrong with them. Blue is a healthy sensor, red are the six failure kinds.

In [ ]:
r = predict_test(ENGINE, PHASE)
true, pred = r["true"].astype(int).ravel(), r["pred"].argmax(-1).ravel()
rec = pd.Series({k: (pred[true == c] == c).mean()
                 for c, k in enumerate(P.SENSOR_FAULT_TYPES) if (true == c).any()})
rec.plot.barh(color=[HEALTHY if k == "none" else FAULTY for k in rec.index], figsize=(7, 3.2))
plt.xlim(0, 1); plt.xlabel("share identified correctly"); plt.title("By sensor condition")
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

### 5. What this means

In [ ]:
faulty = true > 0
print(f"Overall, {(pred == true).mean():.1%} of sensor channels are read correctly - but most "
      f"channels are healthy, so that number flatters it.")
print(f"On channels that really had a fault, it names the kind of fault {(pred[faulty] == true[faulty]).mean():.0%} "
      f"of the time, and spots that the sensor is faulty at all {(pred[faulty] > 0).mean():.0%} of the time.")

---
### Terms used

| Term | Meaning |
|---|---|
| **Window** | 128 seconds of flight data - the unit every model looks at. |
| **Twin** | A perfectly healthy simulated copy of the engine, flown through the exact same flight. |
| **Residual** | Real reading minus the healthy twin's reading. Near zero means the engine behaves as it should. |
| **Train / validation / test** | Flights used to learn / to pick the best version / to score it. A flight is only ever in one of them, so the score is on flights the model never saw. |
| **TBO** | Time between overhauls - the engine's scheduled life: 2000 h, or 1200 h for the 915. |
| **Warm start** | A step begins from what an earlier step already learned instead of from nothing. |